In [1]:
#!/usr/bin/env python3
"""plot_rq2.py — RQ2 privacy-budget figures, one file per panel (for LaTeX subfigures).

    python3 plot_rq2.py  results_rq2_eps  rq2_figs

  In Jupyter: run this cell, then in a new cell
    main(["results_rq2_eps", "rq2_figs"])

Reads rq2_quality.csv, rq2_perf.csv, rq2_epochs.csv written by
RQ2_PrivateBudget (run_rq2.sh).  epsilon = 0 is zero-information placement
(hash of primary / secondary key, placement=random); epsilon > 0 is the
DP data-aware placement (placement=dpart).  The x axis is categorical:
"0 (zero-info)", 0.5, 1, 1.5, 2.

Writes to OUT_DIR, each as .pdf and .png:
  rq2_affinity_cut        affinity cut vs eps (mean +- sd over repeats)        [quality]
  rq2_fanout              partitions touched per txn vs eps (mean +- sd)       [quality]
  rq2_size_cv             partition-size coefficient of variation vs eps       [quality]
  rq2_churn               churn vs noiseless partitioning vs eps (mean +- sd)  [quality]
  rq2_padding             dummy + stash-reserve blocks vs eps (stacked bars)   [quality]
  rq2_throughput          committed txn/s vs eps, without / with refresh       [perf]
  rq2_refresh             mean repartition pause (ms) vs eps; labels = count, % of wall [perf]
  rq2_epoch_throughput    per-epoch committed txn/s, refresh epochs marked     [epochs]
  rq2_ioo_occupancy       per-epoch IOO occupancy (used / capacity)            [epochs]
  rq2_summary.csv         the numbers behind the figures
"""
import argparse, csv, glob, os, sys

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

In [2]:
plt.rcParams.update({"font.size": 8, "axes.labelsize": 8, "legend.fontsize": 7,
                     "xtick.labelsize": 7.5, "ytick.labelsize": 7.5, "pdf.fonttype": 42, "ps.fonttype": 42})

DP_C, DP_C2, RND_C = "#0b4f9c", "#5b8fd0", "#ff7f0e"     # same colours as the RQ3 figures
TEXT, MUTED, GRID = "#1f1f1e", "#6b6a63", "#e6e5df"
EPS_LINE = {0.5: "#9ec2ea", 1.0: "#5b8fd0", 1.5: "#2a6bb8", 2.0: "#0b3f80"}
XLABEL = "Privacy budget $\\epsilon$"

In [3]:

def f(x):
    try:
        return float(x)
    except (TypeError, ValueError):
        return float("nan")


def find(d, name):
    p = os.path.join(d, name)
    if os.path.exists(p):
        return p
    hits = glob.glob(os.path.join(d, "**", name), recursive=True)   # zip unpacked one level deeper
    return hits[0] if hits else None


def read(path):
    if not path:
        return []
    with open(path) as fh:
        return list(csv.DictReader(fh))


def eps_label(e):
    return "0\n(zero-info)" if e <= 0 else f"{e:g}"


def style(ax):
    ax.grid(True, axis="y", color=GRID, linewidth=0.6, zorder=0)
    for s in ("top", "right"):
        ax.spines[s].set_visible(False)
    for s in ("left", "bottom"):
        ax.spines[s].set_color("#bdbcb5")
    ax.tick_params(colors=MUTED)

In [4]:
def main(argv=None):
    ap = argparse.ArgumentParser()
    ap.add_argument("csv_dir")
    ap.add_argument("out")
    ap.add_argument("--width", type=float, default=3.4)
    ap.add_argument("--height", type=float, default=2.4)
    ap.add_argument("--values", action="store_true", help="print the value above every bar / point")
    ap.add_argument("--padding-zero", action="store_true",
                    help="padding figure: also show eps = 0 (zero-information has no padding, so it is hidden by default)")
    a = ap.parse_args(argv)       # argv=None -> command line; in Jupyter pass a list

    qpath, ppath, epath = (find(a.csv_dir, n) for n in ("rq2_quality.csv", "rq2_perf.csv", "rq2_epochs.csv"))
    print("  quality:", qpath, "\n  perf:   ", ppath, "\n  epochs: ", epath)
    Q, P, E = read(qpath), read(ppath), read(epath)
    if not (Q or P or E):
        print("!! no rq2_*.csv in", os.path.abspath(a.csv_dir), " (cwd:", os.getcwd() + ")")
        return
    os.makedirs(a.out, exist_ok=True)
    print("  output ->", os.path.abspath(a.out))

    Q.sort(key=lambda r: f(r["epsilon"]))
    P.sort(key=lambda r: f(r["epsilon"]))

    def save(fig, name):
        fig.tight_layout(pad=0.3)
        for ext in ("pdf", "png"):
            fig.savefig(os.path.join(a.out, f"{name}.{ext}"), dpi=300, bbox_inches="tight")
        plt.close(fig)
        print("  wrote", name + ".pdf/.png")

    def colors(rows):
        return [RND_C if f(r["epsilon"]) <= 0 else DP_C for r in rows]

    def bar_metric(rows, key, sd_key, ylabel, name, fmt="{:.2f}", ylim0=True, ref=None):
        if not rows or key not in rows[0]:
            return
        xs = list(range(len(rows)))
        ys = [f(r[key]) for r in rows]
        es = [f(r[sd_key]) if sd_key and sd_key in r else 0.0 for r in rows]
        fig, ax = plt.subplots(figsize=(a.width, a.height))
        cs = colors(rows)
        bars = ax.bar(xs, ys, yerr=es, color=cs, width=0.62, capsize=2.5, zorder=3,
                      error_kw={"elinewidth": 0.8, "ecolor": TEXT})
        for b, r in zip(bars, rows):
            if f(r["epsilon"]) <= 0:
                b.set_hatch("///"); b.set_edgecolor("white")
        if a.values:
            top = max(y + e for y, e in zip(ys, es)) or 1
            for x, y, e in zip(xs, ys, es):
                ax.text(x, y + e + 0.02 * top, fmt.format(y), ha="center", va="bottom", fontsize=6.5, color=TEXT)
        if ref is not None:
            ax.axhline(ref[0], color=MUTED, linestyle=":", linewidth=0.9)
            ax.text(len(rows) - 0.5, ref[0], ref[1], ha="right", va="bottom", fontsize=6.5, color=MUTED)
        ax.set_xticks(xs, [eps_label(f(r["epsilon"])) for r in rows])
        ax.set_xlabel(XLABEL, color=TEXT); ax.set_ylabel(ylabel, color=TEXT)
        if ylim0:
            ax.set_ylim(bottom=0)
        style(ax)
        h = [plt.Rectangle((0, 0), 1, 1, fc=RND_C, hatch="///", ec="white"),
             plt.Rectangle((0, 0), 1, 1, color=DP_C)]
        ax.legend(h, ["zero-information placement", "SidePart DP placement"], frameon=False, loc="best")
        save(fig, name)

    # ---------------- quality (rq2_quality.csv) ---------------------------
    if Q:
        k = f(Q[0].get("k"))
        bar_metric(Q, "affinity_cut_mean", "affinity_cut_sd", "Affinity cut (fraction)", "rq2_affinity_cut")
        bar_metric(Q, "fanout_mean", "fanout_sd", "Partitions touched per txn", "rq2_fanout",
                   ref=(k, f"all partitions ($k$={k:g})") if k == k else None)
        bar_metric(Q, "size_cv_mean", None, "Partition-size CV", "rq2_size_cv", fmt="{:.3f}")
        bar_metric(Q, "churn_vs_noiseless_mean", "churn_sd", "Churn vs. noiseless partitioning",
                   "rq2_churn", fmt="{:.3f}")

        # padding: dummy blocks and stash reserve, stacked.  eps = 0 (zero-information)
        # has no DP padding by construction, so only the DP budgets are shown.
        QP = Q if a.padding_zero else [r for r in Q if f(r["epsilon"]) > 0]
        xs = list(range(len(QP)))
        dum = [f(r["total_dummy"]) for r in QP]
        stash = [f(r["total_stash_reserve"]) for r in QP]
        fig, ax = plt.subplots(figsize=(a.width, a.height))
        ax.bar(xs, dum, width=0.62, color=DP_C, label="dummy blocks", zorder=3)
        ax.bar(xs, stash, width=0.62, bottom=dum, color=DP_C2, label="stash reserve", zorder=3)
        if a.values:
            for x, d, s in zip(xs, dum, stash):
                ax.text(x, d + s, f"{d + s:g}", ha="center", va="bottom", fontsize=6.5, color=TEXT)
        ax.set_xticks(xs, [eps_label(f(r["epsilon"])) for r in QP])
        ax.set_xlabel(XLABEL, color=TEXT)
        ax.set_ylabel(f"Padding blocks (sum over {Q[0].get('repeats', '?')} repeats)", color=TEXT)
        ax.set_ylim(bottom=0); style(ax)
        ax.legend(frameon=False, loc="upper right")
        save(fig, "rq2_padding")

    # ---------------- performance (rq2_perf.csv) --------------------------
    if P:
        xs = list(range(len(P)))
        t0 = [f(r["throughput_txn_s"]) for r in P]
        t1 = [f(r["throughput_txn_s_with_refresh"]) for r in P]
        fig, ax = plt.subplots(figsize=(a.width, a.height))
        w = 0.36
        b0 = ax.bar([x - w / 2 for x in xs], t0, width=w, color=colors(P), zorder=3, label="epochs only")
        b1 = ax.bar([x + w / 2 for x in xs], t1, width=w, color=colors(P), alpha=0.45, zorder=3,
                    label="incl. repartition pauses")
        for bb, r in zip(zip(b0, b1), P):
            if f(r["epsilon"]) <= 0:
                for b in bb:
                    b.set_hatch("///"); b.set_edgecolor("white")
        if a.values:
            for x, y0, y1 in zip(xs, t0, t1):
                ax.text(x - w / 2, y0, f"{y0:.0f}", ha="center", va="bottom", fontsize=5.5, color=TEXT, rotation=90)
                ax.text(x + w / 2, y1, f"{y1:.0f}", ha="center", va="bottom", fontsize=5.5, color=TEXT, rotation=90)
        ax.set_xticks(xs, [eps_label(f(r["epsilon"])) for r in P])
        ax.set_xlabel(XLABEL, color=TEXT); ax.set_ylabel("Committed throughput (txn/s)", color=TEXT)
        ax.set_ylim(0, max(t0 + t1) * 1.15); style(ax)
        h = [plt.Rectangle((0, 0), 1, 1, color=DP_C), plt.Rectangle((0, 0), 1, 1, color=DP_C, alpha=0.45),
             plt.Rectangle((0, 0), 1, 1, fc=RND_C, hatch="///", ec="white")]
        ax.legend(h, ["epochs only", "incl. repartition pauses", "zero-information placement"],
                  frameon=False, loc="lower center", bbox_to_anchor=(0.5, 1.0), ncol=2, fontsize=6.5)
        save(fig, "rq2_throughput")

        # refresh cost: mean pause per repartition; label = #repartitions and % of wall time
        pause = [f(r["mean_refresh_pause_ms"]) for r in P]
        pct = [f(r["amortized_refresh_pct"]) for r in P]
        nref = [f(r["refreshes"]) for r in P]
        fig, ax = plt.subplots(figsize=(a.width, a.height))
        ax.bar(xs, pause, width=0.55, color=colors(P), zorder=3)
        for x, p, n, q in zip(xs, pause, nref, pct):
            ax.text(x, p, f"{n:g}x\n{q:.1f}%", ha="center", va="bottom", fontsize=6.5, color=TEXT)
        ax.set_ylabel("Mean repartition pause (ms)", color=TEXT)
        ax.set_ylim(0, max(pause + [1]) * 1.3)
        ax.set_xticks(xs, [eps_label(f(r["epsilon"])) for r in P])
        ax.set_xlabel(XLABEL, color=TEXT)
        ax.text(0.01, 0.98, "labels: #repartitions / % of wall time", transform=ax.transAxes,
                ha="left", va="top", fontsize=6.5, color=MUTED)
        style(ax)
        save(fig, "rq2_refresh")

    # ---------------- per-epoch trace (rq2_epochs.csv) --------------------
    if E:
        series = {}
        for r in E:
            series.setdefault(round(f(r["epsilon"]), 4), []).append(r)
        for key, ylabel, name, scale in (("throughput_txn_s", "Committed throughput (txn/s)", "rq2_epoch_throughput", 1),
                                         ("ioo_max_occupancy", "IOO occupancy (used / capacity)", "rq2_ioo_occupancy", 1)):
            fig, ax = plt.subplots(figsize=(a.width, a.height))
            for e in sorted(series):
                rr = sorted(series[e], key=lambda r: int(f(r["epoch"])))
                xs = [int(f(r["epoch"])) for r in rr]
                ys = [f(r[key]) * scale for r in rr]
                if e <= 0:
                    kw = dict(color=RND_C, linestyle="--", marker="s", markerfacecolor="white",
                              label="$\\epsilon$=0 (zero-information)")
                else:
                    kw = dict(color=EPS_LINE.get(e, DP_C), linestyle="-", marker="o",
                              label=f"$\\epsilon$={e:g}")
                ax.plot(xs, ys, linewidth=1.2, markersize=3.5, zorder=3, **kw)
                for x, y, r in zip(xs, ys, rr):
                    if f(r.get("refreshed")) >= 1:
                        ax.plot([x], [y], marker="*", markersize=9, color=kw["color"],
                                markeredgecolor=TEXT, markeredgewidth=0.5, zorder=4)
            ax.plot([], [], marker="*", linestyle="", color="white", markeredgecolor=TEXT,
                    markersize=8, label="repartition after epoch")
            ax.set_xlabel("Epoch", color=TEXT); ax.set_ylabel(ylabel, color=TEXT)
            ax.set_ylim(0, None if key != "ioo_max_occupancy" else 1.0)
            if key == "throughput_txn_s":
                top = max(f(r[key]) for r in E)
                ax.set_ylim(0, top * 1.5)
            style(ax)
            ax.legend(frameon=False, ncol=3, fontsize=6, loc="upper center")
            save(fig, name)

    # ---------------- numbers behind the figures --------------------------
    with open(os.path.join(a.out, "rq2_summary.csv"), "w", newline="") as fh:
        w = csv.writer(fh)
        w.writerow(["epsilon", "placement", "affinity_cut_mean", "affinity_cut_sd", "fanout_mean", "size_cv_mean",
                    "churn_mean", "total_dummy", "total_stash_reserve", "throughput_txn_s",
                    "throughput_txn_s_with_refresh", "refreshes", "mean_refresh_pause_ms", "amortized_refresh_pct"])
        pmap = {round(f(r["epsilon"]), 4): r for r in P}
        for r in Q or P:
            e = round(f(r["epsilon"]), 4)
            p = pmap.get(e, {})
            w.writerow([e, r.get("placement", ""), r.get("affinity_cut_mean", ""), r.get("affinity_cut_sd", ""),
                        r.get("fanout_mean", ""), r.get("size_cv_mean", ""), r.get("churn_vs_noiseless_mean", ""),
                        r.get("total_dummy", ""), r.get("total_stash_reserve", ""), p.get("throughput_txn_s", ""),
                        p.get("throughput_txn_s_with_refresh", ""), p.get("refreshes", ""),
                        p.get("mean_refresh_pause_ms", ""), p.get("amortized_refresh_pct", "")])
    print("  wrote rq2_summary.csv")


if __name__ == "__main__" and "ipykernel" not in sys.modules:
    main()


In [ ]:
main(["results_rq2_eps", "rq2_figs"])